In [1]:
import sys; print(sys.executable) 

/home/cjen/mySageMaker/ML/spark-nlp/redaction/dat/.venv/bin/python


In [2]:
import boto3

region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)

In [3]:
from sagemaker import Session, s3

session = Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml


In [4]:
import boto3

boto_session = boto3.Session()

region = boto_session.region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account}"

inputs = s3.S3Uploader.upload("data", "s3://{}/spark-nlp/redaction/jgh-msss-prem/redacted_input".format(bucket))

print("S3 location " + inputs)

S3 location s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_input


In [5]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [6]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp" \
    "spark-nlp-display" \
    "boto3"

print("All packages installed.")

All packages installed.


In [7]:
import sagemaker
from sagemaker import get_execution_role

sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region = boto_session.region_name or sm_session.boto_region_name
bucket = sm_session.default_bucket()
prefix = "spark-nlp/redaction"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

SageMaker SDK version : 2.257.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


In [8]:
# ── Verify this notebook is running within a SageMaker environment ─────────────
import os, json, urllib.request

print("=" * 60)
print("  SageMaker Session Proof")
print("=" * 60)

# 1. Active SageMaker SDK session object
print(f"\n[SDK] sagemaker.__version__  : {sagemaker.__version__}")
print(f"[SDK] Session boto region    : {sm_session.boto_region_name}")
print(f"[SDK] Default S3 bucket      : {sm_session.default_bucket()}")
print(f"[SDK] Execution role ARN     : {role}")
print(f"[SDK] Boto3 caller identity  : {boto3.client('sts').get_caller_identity()['Arn']}")

# 2. SageMaker-specific env vars (set automatically by SageMaker runtimes)
sm_keys = sorted(k for k in os.environ if k.startswith("SM_") or "SAGEMAKER" in k.upper())
if sm_keys:
    print(f"\n[ENV] SageMaker env vars found ({len(sm_keys)}):")
    for k in sm_keys:
        print(f"      {k} = {os.environ[k]}")
else:
    print("\n[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.")

# 3. /opt/ml/metadata/resource-metadata.json
#    Present in Studio domains, Processing jobs, and Training jobs.
meta_path = "/opt/ml/metadata/resource-metadata.json"
if os.path.exists(meta_path):
    with open(meta_path) as fh:
        meta = json.load(fh)
    print(f"\n[META] {meta_path}:")
    print(json.dumps(meta, indent=4))
else:
    print(f"\n[META] {meta_path} not found (not inside a managed job container).")

# 4. EC2 IMDSv2 endpoint — reachable on SageMaker Notebook Instances and job instances.
#    If this prints an instance type, the notebook is running on a real SageMaker host.
_imds = "http://169.254.169.254/latest"
try:
    token = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/api/token",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            method="PUT",
        ), timeout=1
    ).read().decode()
    inst_type = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/meta-data/instance-type",
            headers={"X-aws-ec2-metadata-token": token},
        ), timeout=1
    ).read().decode()
    print(f"\n[IMDS] EC2 instance type : {inst_type}  ← confirms managed SageMaker host")
except Exception:
    print("\n[IMDS] EC2 IMDS not reachable — expected when running locally.")
    print("       On a real SageMaker Notebook Instance the instance type would appear here.")

print("\n✓ SageMaker SDK session is authenticated and active.")

  SageMaker Session Proof

[SDK] sagemaker.__version__  : 2.257.1
[SDK] Session boto region    : us-west-2
[SDK] Default S3 bucket      : sagemaker-us-west-2-493644444178
[SDK] Execution role ARN     : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd
[SDK] Boto3 caller identity  : arn:aws:sts::493644444178:assumed-role/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd/mindy@theclinician.com

[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.

[META] /opt/ml/metadata/resource-metadata.json not found (not inside a managed job container).

[IMDS] EC2 IMDS not reachable — expected when running locally.
       On a real SageMaker Notebook Instance the instance type would appear here.

✓ SageMaker SDK session is authenticated and active.


In [9]:
import sys
print("Python executable:", sys.executable)

import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ArrayType

import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")
print("Spark NLP version:", sparknlp.version())

Python executable: /home/cjen/mySageMaker/ML/spark-nlp/redaction/dat/.venv/bin/python
Spark NLP version: 6.3.3


In [10]:
params = {
    "spark.driver.memory"           : "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize"    : "2000M",
    "spark.ui.enabled"              : "false",
}

spark = sparknlp.start(params=params)
spark

26/03/24 09:34:53 WARN Utils: Your hostname, MindyJen1008 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/24 09:34:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/cjen/mySageMaker/ML/customer_sentiment/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/cjen/.ivy2/cache
The jars for the packages stored in: /home/cjen/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-50687396-a133-430f-8715-06fad1c1e264;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	f

In [ ]:
# ── Download input data from S3 → local /tmp ──────────────────────────────────
import os, glob
from sagemaker.s3 import S3Downloader

# S3 path set during the upload cell above
s3_input_path = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_input"
local_data_dir = "/tmp/redaction_input"
os.makedirs(local_data_dir, exist_ok=True)

print(f"Source : {s3_input_path}")
print(f"Dest   : {local_data_dir}\n")

# S3Downloader is the SageMaker v2.x-compatible helper (no extra IAM policy needed)
S3Downloader.download(s3_input_path, local_data_dir, sagemaker_session=sm_session)

downloaded = [f for f in glob.glob(os.path.join(local_data_dir, "**/*"), recursive=True)
              if os.path.isfile(f)]
print(f"Downloaded {len(downloaded)} file(s):")
for f in downloaded:
    print(f"  {f}")

In [ ]:
# ── Read Excel → pandas → Spark DataFrame ─────────────────────────────────────
import glob
import pandas as pd

# Locate the downloaded Excel file(s)
xlsx_files = sorted(
    glob.glob(os.path.join(local_data_dir, "**/*.xlsx"), recursive=True)
    + glob.glob(os.path.join(local_data_dir, "*.xlsx"))
)

if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files found under {local_data_dir}")

print(f"Reading : {xlsx_files[0]}\n")

# pandas read — dtype=str keeps all values as text (important for clinical notes)
pdf = pd.read_excel(xlsx_files[0], dtype=str).fillna("")

print("── pandas preview " + "─" * 44)
print(f"Shape   : {pdf.shape}  ({pdf.shape[0]} rows × {pdf.shape[1]} cols)")
print(f"Columns : {list(pdf.columns)}\n")
print(pdf.head(3).to_string(index=False))

# Convert to Spark DataFrame
sdf = spark.createDataFrame(pdf)

print("\n── Spark DataFrame schema " + "─" * 36)
sdf.printSchema()

print("── First 5 rows (truncated at 80 chars) " + "─" * 21)
sdf.show(5, truncate=80)
print(f"Total rows : {sdf.count()}")